
# 07 – Merge de fuentes

Este notebook construye un dataset analítico con llave:

`(cow_id, date)`

Integra las siguientes fuentes:

- `milking.parquet`
- `rumination.parquet`
- `events_daily.parquet`
- `feeding_daily_corral.parquet`
- `weather.parquet`
- `diet_period_summary.parquet`
- `diet_periods.parquet`

## Idea general

- **Milking, rumination y events_daily** ya viven a nivel vaca-fecha o se agregan a ese nivel.
- **Weather** vive a nivel fecha y se replica a todas las vacas del mismo día.
- **Feeding** vive a nivel fecha-corral. Como no existe un mapeo confiable `cow_id -> corral` por fecha, se agrega **por fecha**.
- **Diet summary** y **diet periods** viven por periodo, así que se expanden a diario.


## 1. Importaciones

In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


## 2. Rutas

In [2]:

def resolve_path(filename: str) -> Path:
    candidates = [
        Path(filename),
        Path("./") / filename,
        Path("/mnt/data") / filename,
        Path("../data/interim") / filename,
        Path("../data/processed") / filename,
        Path("outputs") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f"No se encontró {filename}")

PATH_MILKING = resolve_path("milking.parquet")
PATH_RUMINATION = resolve_path("rumination.parquet")
PATH_EVENTS_DAILY = resolve_path("events_daily.parquet")
PATH_FEEDING = resolve_path("feeding_daily_corral.parquet")
PATH_WEATHER = resolve_path("weather.parquet")
PATH_DIET_SUMMARY = resolve_path("diet_period_summary.parquet")
PATH_DIET_PERIODS = resolve_path("diet_periods.parquet")

OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_DATASET = OUT_DIR / "training_dataset.parquet"
OUT_MISSING = OUT_DIR / "training_dataset_missing_report.csv"

print("Milking      :", PATH_MILKING)
print("Rumination   :", PATH_RUMINATION)
print("Events Daily :", PATH_EVENTS_DAILY)
print("Feeding      :", PATH_FEEDING)
print("Weather      :", PATH_WEATHER)
print("Diet Summary :", PATH_DIET_SUMMARY)
print("Diet Periods :", PATH_DIET_PERIODS)
print("Output       :", OUT_DATASET.resolve())


Milking      : /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/interim/milking.parquet
Rumination   : /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/interim/rumination.parquet
Events Daily : /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/interim/events_daily.parquet
Feeding      : /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/interim/feeding_daily_corral.parquet
Weather      : /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/interim/weather.parquet
Diet Summary : /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/interim/diet_period_summary.parquet
Diet Periods : /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/interim/diet_periods.parquet
Output       : /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/processed/training_dataset.parquet


## 3. Carga segura de Parquet

In [3]:

def read_parquet_safe(path: Path) -> pd.DataFrame:
    table = pq.read_table(path)
    return table.to_pandas()

milking_raw = read_parquet_safe(PATH_MILKING)
rumination_raw = read_parquet_safe(PATH_RUMINATION)
events_daily_raw = read_parquet_safe(PATH_EVENTS_DAILY)
feeding_raw = read_parquet_safe(PATH_FEEDING)
weather_raw = read_parquet_safe(PATH_WEATHER)
diet_summary_raw = read_parquet_safe(PATH_DIET_SUMMARY)
diet_periods_raw = read_parquet_safe(PATH_DIET_PERIODS)

for name, df in {
    "milking_raw": milking_raw,
    "rumination_raw": rumination_raw,
    "events_daily_raw": events_daily_raw,
    "feeding_raw": feeding_raw,
    "weather_raw": weather_raw,
    "diet_summary_raw": diet_summary_raw,
    "diet_periods_raw": diet_periods_raw,
}.items():
    print(f"{name:20s} -> {df.shape}")


milking_raw          -> (23763, 17)
rumination_raw       -> (14654, 6)
events_daily_raw     -> (4033, 80)
feeding_raw          -> (1626, 12)
weather_raw          -> (41616, 7)
diet_summary_raw     -> (3, 18)
diet_periods_raw     -> (20, 10)


## 4. Diagnóstico rápido

In [4]:

for name, df in {
    "milking_raw": milking_raw,
    "rumination_raw": rumination_raw,
    "events_daily_raw": events_daily_raw,
    "feeding_raw": feeding_raw,
    "weather_raw": weather_raw,
    "diet_summary_raw": diet_summary_raw,
    "diet_periods_raw": diet_periods_raw,
}.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print(df.dtypes.head(20))
    display(df.head(3))



milking_raw
numero_ordeno              int64
duracion_mmss                str
produccion_kg            float64
intervalo_ordeno_hhmm        str
di                       float64
dd                       float64
td                       float64
ubre                       int64
pezon                        str
destino_leche                str
ms                           str
cow_id                     int64
source_file                  str
duracion_min             float64
intervalo_ordeno_min     float64
ti                       float64
date                      object
dtype: object


,numero_ordeno,duracion_mmss,produccion_kg,intervalo_ordeno_hhmm,di,dd,td,ubre,pezon,destino_leche,ms,cow_id,source_file,duracion_min,intervalo_ordeno_min,ti,date
0,1,06:22,16.72,10:56,4.44,5.57,6.71,0,NaN,Tanque,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN,NaN,2025-01-01
1,2,04:58,14.25,08:46,4.13,4.85,5.27,0,NaN,Tanque,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN,NaN,2025-01-01
2,3,05:53,16.48,11:13,5.76,4.11,6.61,0,NaN,Tanque,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN,NaN,2025-01-01



rumination_raw
cow_id                         int64
group                          int64
last_calving          datetime64[us]
days_in_milk                   int64
date                  datetime64[us]
ruminating_minutes             int64
dtype: object


,cow_id,group,last_calving,days_in_milk,date,ruminating_minutes
1,2110,1,2025-01-12,167,2025-06-28,595
2,2110,1,2025-01-12,168,2025-06-29,569
3,2110,1,2025-01-12,169,2025-06-30,490



events_daily_raw
cow_id                            int64
date                     datetime64[us]
event_count                       int64
event_types                         str
raw_event_text                      str
diagnosis_text                      str
medication_text                     str
treatment_text                      str
diagnosis_categories                str
treatment_categories                str
medication_categories               str
body_condition_score            float64
weight_kg                       float64
liters_at_dryoff                float64
dcc_at_dryoff                   float64
del_at_dryoff                   float64
gestant_insem_number            float64
pregnancy_positive                int64
pregnancy_negative                int64
mastitis_flag                     int64
dtype: object


...

,date,time,temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m
0,2021-04-03,2021-04-03 00:00:00,10.2,84,1023.9,0.0,19.5
1,2021-04-03,2021-04-03 01:00:00,9.8,85,1023.7,0.0,18.4
2,2021-04-03,2021-04-03 02:00:00,9.4,86,1023.5,0.0,16.2



diet_summary_raw
diet_period                     str
period_start         datetime64[us]
period_end           datetime64[us]
consumo_kg_vaca               int64
am_pm_fraction              float64
n_ingredientes                int64
diet_dm_total               float64
diet_wet_total              float64
diet_pct_ms_total           float64
usa_oro_milk                  int64
usa_oro_balance               int64
usa_silo_maiz                 int64
usa_silo_avena                int64
usa_ensilado                  int64
usa_heno                      int64
usa_triticale                 int64
usa_melaza                    int64
usa_soya                      int64
dtype: object


,diet_period,period_start,period_end,consumo_kg_vaca,am_pm_fraction,n_ingredientes,diet_dm_total,diet_wet_total,diet_pct_ms_total,usa_oro_milk,usa_oro_balance,usa_silo_maiz,usa_silo_avena,usa_ensilado,usa_heno,usa_triticale,usa_melaza,usa_soya
0,dieta Mayo,2025-01-01,2025-05-31,154,0.5,5,23.7259,50.21,2.491,1,0,0,0,0,0,1,1,0
1,Dieta Junio,2025-06-01,2025-07-31,164,0.5,6,23.4909,52.45,3.361,1,1,0,0,0,0,1,1,0
2,Dieta Agosto,2025-08-01,2025-09-30,160,0.5,9,24.7834,54.02,5.471,1,1,1,1,0,0,0,1,1



diet_periods_raw
ingrediente                   str
pct_ms                    float64
kg_humeda                 float64
kg_seca                   float64
kg_am_pm                  float64
diet_period                   str
period_start       datetime64[us]
period_end         datetime64[us]
consumo_kg_vaca             int64
am_pm_fraction            float64
dtype: object


,ingrediente,pct_ms,kg_humeda,kg_seca,kg_am_pm,diet_period,period_start,period_end,consumo_kg_vaca,am_pm_fraction
0,Agua,0.001,5.50,0.0055,423.50,dieta Mayo,2025-01-01,2025-05-31,154,0.5
1,Melaza(50% Agua),0.400,3.33,1.3320,256.41,dieta Mayo,2025-01-01,2025-05-31,154,0.5
2,ORO MILK,0.900,15.76,14.1840,1213.52,dieta Mayo,2025-01-01,2025-05-31,154,0.5


## 5. Utilidades de limpieza

In [5]:

def normalize_date_col(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df = df.copy()
    df[col] = pd.to_datetime(df[col], errors="coerce").dt.normalize()
    return df

def to_numeric_if_exists(df: pd.DataFrame, columns):
    df = df.copy()
    for c in columns:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def parse_mmss_to_minutes(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not s or s.lower() == "nan":
        return np.nan
    if ":" not in s:
        return pd.to_numeric(s, errors="coerce")
    parts = s.split(":")
    try:
        if len(parts) == 2:
            mm, ss = parts
            return int(mm) + int(ss) / 60.0
        if len(parts) == 3:
            hh, mm, ss = parts
            return int(hh) * 60.0 + int(mm) + int(ss) / 60.0
    except Exception:
        return np.nan
    return np.nan

def parse_hhmm_to_minutes(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not s or s.lower() == "nan":
        return np.nan
    if ":" not in s:
        return pd.to_numeric(s, errors="coerce")
    parts = s.split(":")
    try:
        if len(parts) == 2:
            hh, mm = parts
            return int(hh) * 60.0 + int(mm)
        if len(parts) == 3:
            hh, mm, ss = parts
            return int(hh) * 60.0 + int(mm) + int(ss) / 60.0
    except Exception:
        return np.nan
    return np.nan

def sanitize_name(x):
    x = str(x).strip().lower()
    repl = {
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", "ñ": "n",
        " ": "_", "-": "_", "/": "_", "(": "", ")": "", "%": "pct",
        ".": "", ",": "", ":": "", ";": "", "+": "_plus_"
    }
    for a, b in repl.items():
        x = x.replace(a, b)
    while "__" in x:
        x = x.replace("__", "_")
    return x.strip("_")


## 6. Normalización de fechas y tipos

In [6]:

milking_raw = normalize_date_col(milking_raw, "date")
rumination_raw = normalize_date_col(rumination_raw, "date")
events_daily_raw = normalize_date_col(events_daily_raw, "date")
feeding_raw = normalize_date_col(feeding_raw, "date")
weather_raw = normalize_date_col(weather_raw, "date")

for col in ["period_start", "period_end"]:
    if col in diet_summary_raw.columns:
        diet_summary_raw[col] = pd.to_datetime(diet_summary_raw[col], errors="coerce").dt.normalize()
    if col in diet_periods_raw.columns:
        diet_periods_raw[col] = pd.to_datetime(diet_periods_raw[col], errors="coerce").dt.normalize()

milking_raw = to_numeric_if_exists(
    milking_raw,
    ["cow_id", "numero_ordeno", "produccion_kg", "di", "dd", "td", "ubre", "duracion_min", "intervalo_ordeno_min", "ti"]
)
rumination_raw = to_numeric_if_exists(
    rumination_raw,
    ["cow_id", "group", "days_in_milk", "ruminating_minutes"]
)
events_daily_raw = to_numeric_if_exists(
    events_daily_raw,
    ["cow_id", "event_count"]
)
feeding_raw = to_numeric_if_exists(
    feeding_raw,
    ["year", "month", "day", "corral", "kg_am", "kg_pm", "kg_totales", "sobrante", "consumo", "rechazo"]
)

for df_name in ["milking_raw", "rumination_raw", "events_daily_raw"]:
    df = globals()[df_name]
    if "cow_id" in df.columns:
        df["cow_id"] = df["cow_id"].astype("Int64")
    globals()[df_name] = df

if "corral" in feeding_raw.columns:
    feeding_raw["corral"] = feeding_raw["corral"].astype("Int64")

# Completar minutos si vienen vacíos pero existe la versión en texto
if "duracion_min" in milking_raw.columns and "duracion_mmss" in milking_raw.columns:
    missing = milking_raw["duracion_min"].isna()
    milking_raw.loc[missing, "duracion_min"] = milking_raw.loc[missing, "duracion_mmss"].map(parse_mmss_to_minutes)

if "intervalo_ordeno_min" in milking_raw.columns and "intervalo_ordeno_hhmm" in milking_raw.columns:
    missing = milking_raw["intervalo_ordeno_min"].isna()
    milking_raw.loc[missing, "intervalo_ordeno_min"] = milking_raw.loc[missing, "intervalo_ordeno_hhmm"].map(parse_hhmm_to_minutes)

print("Normalización completada.")


Normalización completada.


## 7. Construcción de tablas diarias

### 7.1 Ordeño diario por vaca

In [7]:
agg_map = {
    "ordenos_dia": ("numero_ordeno", "count"),
    "produccion_kg": ("produccion_kg", "sum"),
    "duracion_total_min": ("duracion_min", "sum"),
    "duracion_prom_min": ("duracion_min", "mean"),
    "intervalo_ordeno_prom_min": ("intervalo_ordeno_min", "mean"),
    "di_mean": ("di", "mean"),
    "dd_mean": ("dd", "mean"),
    "td_mean": ("td", "mean"),
    "ti_mean": ("ti", "mean"),
}

# columnas opcionales
optional_first = [
    "ubre",
    "destino_leche",
    "ms",
    "source_file",
]

optional_join = [
    "razon_desviacion"
]

for col in optional_first:
    if col in milking_raw.columns:
        agg_map[col] = (col, "first")

for col in optional_join:
    if col in milking_raw.columns:
        agg_map[col] = (
            col,
            lambda s: " | ".join(sorted(pd.Series(s).dropna().astype(str).unique()))
        )

# dejar solo columnas que realmente existan
agg_map = {
    new_col: (src_col, func)
    for new_col, (src_col, func) in agg_map.items()
    if src_col in milking_raw.columns
}

milking_daily = (
    milking_raw
    .dropna(subset=["cow_id", "date"])
    .groupby(["cow_id", "date"], as_index=False)
    .agg(**agg_map)
)

print("milking_daily ->", milking_daily.shape)
display(milking_daily.head())

milking_daily -> (9814, 15)


,cow_id,date,ordenos_dia,produccion_kg,duracion_total_min,duracion_prom_min,intervalo_ordeno_prom_min,di_mean,dd_mean,td_mean,ti_mean,ubre,destino_leche,ms,source_file
0,1204,2025-01-01,1,16.68,11.216667,11.216667,1309.0,5.160,4.64,6.880,0.000,0,Tanque,VMS 1,Producciones de leche1204.xls
1,1204,2025-01-02,2,17.91,14.516667,7.258333,802.0,1.325,2.22,2.535,2.875,1,Divert 3,VMS 1,Producciones de leche1204.xls
2,1204,2025-01-03,2,25.36,14.683333,7.341667,691.0,3.130,3.26,4.905,1.385,1,Tanque,VMS 1,Producciones de leche1204.xls
3,1204,2025-01-04,1,16.71,8.466667,8.466667,700.0,4.130,3.89,5.730,2.960,1,Tanque,VMS 1,Producciones de leche1204.xls
4,1204,2025-01-05,2,25.04,21.366667,10.683333,1141.0,2.925,2.97,4.200,2.425,1,Tanque,VMS 1,Producciones de leche1204.xls


### 7.2 Rumia diaria por vaca

In [8]:

rumination_daily = (
    rumination_raw
    .dropna(subset=["cow_id", "date"])
    .groupby(["cow_id", "date"], as_index=False)
    .agg(
        rumia_min=("ruminating_minutes", "sum"),
        days_in_milk=("days_in_milk", "max"),
        group_id=("group", "first"),
        last_calving=("last_calving", "first"),
    )
)

print("rumination_daily ->", rumination_daily.shape)
display(rumination_daily.head())


rumination_daily -> (5750, 6)


,cow_id,date,rumia_min,days_in_milk,group_id,last_calving
0,1204,2025-06-28,1314,418,100,2024-05-06
1,1204,2025-06-29,990,419,100,2024-05-06
2,1204,2025-06-30,1098,420,100,2024-05-06
3,1204,2025-07-01,1132,421,100,2024-05-06
4,1204,2025-07-02,1186,422,100,2024-05-06


### 7.3 Eventos diarios por vaca

In [9]:

events_daily = events_daily_raw.copy()

# Asegurar unicidad por cow_id + date
dup_events = events_daily.duplicated(subset=["cow_id", "date"]).sum()
print("Duplicados en events_daily:", dup_events)

if dup_events > 0:
    binary_cols = [
        c for c in events_daily.columns
        if c not in ["cow_id", "date", "event_types", "raw_event_text", "diagnosis_text", "medication_text", "treatment_text",
                     "diagnosis_categories", "treatment_categories", "medication_categories"]
        and pd.api.types.is_numeric_dtype(events_daily[c])
    ]

    text_join = lambda s: " | ".join(sorted(pd.Series(s).dropna().astype(str).replace("", pd.NA).dropna().unique()))

    agg_map = {c: "max" for c in binary_cols}
    agg_map.update({
        "event_count": "sum",
        "event_types": text_join,
        "raw_event_text": text_join,
        "diagnosis_text": text_join,
        "medication_text": text_join,
        "treatment_text": text_join,
        "diagnosis_categories": text_join,
        "treatment_categories": text_join,
        "medication_categories": text_join,
        "body_condition_score": "max",
        "weight_kg": "max",
        "liters_at_dryoff": "max",
        "dcc_at_dryoff": "max",
        "del_at_dryoff": "max",
        "gestant_insem_number": "max",
    })

    events_daily = (
        events_daily
        .groupby(["cow_id", "date"], as_index=False)
        .agg(agg_map)
    )

print("events_daily ->", events_daily.shape)
display(events_daily.head())


Duplicados en events_daily: 0
events_daily -> (4033, 80)


,cow_id,date,event_count,event_types,raw_event_text,diagnosis_text,medication_text,treatment_text,diagnosis_categories,treatment_categories,...,trt_vacuna,trt_vitamina,med_antibiotico,med_bolo_rumensin,med_cefa,med_cidr,med_gnrh,med_other,med_pg,med_vacuna
0,1204,2022-06-05,1,Entrada,User1,,,,,,...,0,0,0,0,0,0,0,0,0,0
1,1204,2022-06-08,1,Cambio ID,User1 Nacimiento,,,,,,...,0,0,0,0,0,0,0,0,0,0
2,1204,2022-07-01,1,Cambio ID,User1 2003 -> 99980,,,,,,...,0,0,0,0,0,0,0,0,0,0
3,1204,2022-08-15,1,Diagnósticos/Tr,SHOT; Med: DelproClien 99980 -> 1204,,DelproClien 99980 -> 1204,,,,...,0,0,0,0,0,0,0,1,0,0
4,1204,2022-09-27,2,Diagnósticos/Tr | Peso,DelproClien Dns: VACUNA; Loc.: DD; Trat: ONE- ...,VACUNA,,ONE,diag_vacuna,trt_antibiotico,...,0,0,0,0,0,0,0,0,0,0


### 7.4 Clima diario por fecha

In [10]:

weather_daily = (
    weather_raw
    .dropna(subset=["date"])
    .groupby("date", as_index=False)
    .agg(
        temperature_2m_mean=("temperature_2m", "mean"),
        temperature_2m_min=("temperature_2m", "min"),
        temperature_2m_max=("temperature_2m", "max"),
        relative_humidity_2m_mean=("relative_humidity_2m", "mean"),
        pressure_msl_mean=("pressure_msl", "mean"),
        precipitation_sum=("precipitation", "sum"),
        wind_speed_10m_mean=("wind_speed_10m", "mean"),
    )
)

print("weather_daily ->", weather_daily.shape)
display(weather_daily.head())


weather_daily -> (1734, 8)


,date,temperature_2m_mean,temperature_2m_min,temperature_2m_max,relative_humidity_2m_mean,pressure_msl_mean,precipitation_sum,wind_speed_10m_mean
0,2021-04-03,15.479167,7.6,26.6,58.083333,1020.004167,0.0,14.341667
1,2021-04-04,15.012500,7.1,26.0,55.791667,1018.475000,0.0,13.808333
2,2021-04-05,16.416667,6.8,28.0,51.708333,1015.008333,0.0,9.283333
3,2021-04-06,17.237500,9.8,28.5,57.083333,1014.283333,0.9,8.433333
4,2021-04-07,19.404167,10.5,29.2,46.375000,1013.375000,0.0,8.245833


### 7.5 Alimentación diaria global por fecha

In [11]:

# Como no existe un mapeo vaca->corral por fecha, esta fuente se resume por fecha y se replica a todas las vacas del mismo día.
feeding_daily = (
    feeding_raw
    .dropna(subset=["date"])
    .groupby("date", as_index=False)
    .agg(
        n_corrales=("corral", "nunique"),
        kg_am_total=("kg_am", "sum"),
        kg_pm_total=("kg_pm", "sum"),
        kg_totales_total=("kg_totales", "sum"),
        sobrante_total=("sobrante", "sum"),
        consumo_total=("consumo", "sum"),
        rechazo_total=("rechazo", "sum"),
        kg_am_prom_corral=("kg_am", "mean"),
        kg_pm_prom_corral=("kg_pm", "mean"),
        kg_totales_prom_corral=("kg_totales", "mean"),
        consumo_prom_corral=("consumo", "mean"),
    )
)

feeding_daily["ratio_consumo_oferta_global"] = (
    feeding_daily["consumo_total"] / feeding_daily["kg_totales_total"].replace(0, np.nan)
)
feeding_daily["ratio_sobrante_oferta_global"] = (
    feeding_daily["sobrante_total"] / feeding_daily["kg_totales_total"].replace(0, np.nan)
)
feeding_daily["ratio_rechazo_consumo_global"] = (
    feeding_daily["rechazo_total"] / feeding_daily["consumo_total"].replace(0, np.nan)
)

print("feeding_daily ->", feeding_daily.shape)
display(feeding_daily.head())


feeding_daily ->

 (271, 15)


,date,n_corrales,kg_am_total,kg_pm_total,kg_totales_total,sobrante_total,consumo_total,rechazo_total,kg_am_prom_corral,kg_pm_prom_corral,kg_totales_prom_corral,consumo_prom_corral,ratio_consumo_oferta_global,ratio_sobrante_oferta_global,ratio_rechazo_consumo_global
0,2025-01-01,6,4000.0,4000.0,8000,10.0,7990,0.769231,666.666667,666.666667,1333.333333,1331.666667,0.998750,0.001250,0.000096
1,2025-01-02,6,3950.0,3950.0,7900,710.0,7190,57.200855,658.333333,658.333333,1316.666667,1198.333333,0.910127,0.089873,0.007956
2,2025-01-03,6,3950.0,3900.0,7850,275.0,7575,21.266789,658.333333,650.000000,1308.333333,1262.500000,0.964968,0.035032,0.002807
3,2025-01-04,6,3950.0,3950.0,7900,395.0,7505,31.079060,658.333333,658.333333,1316.666667,1250.833333,0.950000,0.050000,0.004141
4,2025-01-05,6,3950.0,3950.0,7900,505.0,7395,39.883450,658.333333,658.333333,1316.666667,1232.500000,0.936076,0.063924,0.005393


### 7.6 Dieta diaria expandida desde `diet_period_summary`

In [12]:

diet_summary_daily_parts = []

for _, row in diet_summary_raw.iterrows():
    start = row.get("period_start")
    end = row.get("period_end")
    if pd.isna(start) or pd.isna(end):
        continue

    dates = pd.date_range(start, end, freq="D")
    tmp = pd.DataFrame({"date": dates})

    for c in diet_summary_raw.columns:
        if c not in ["period_start", "period_end"]:
            tmp[c] = row[c]

    diet_summary_daily_parts.append(tmp)

diet_summary_daily = (
    pd.concat(diet_summary_daily_parts, ignore_index=True)
    if diet_summary_daily_parts
    else pd.DataFrame(columns=["date"])
)

print("diet_summary_daily ->", diet_summary_daily.shape)
display(diet_summary_daily.head())


diet_summary_daily -> (273, 17)


,date,diet_period,consumo_kg_vaca,am_pm_fraction,n_ingredientes,diet_dm_total,diet_wet_total,diet_pct_ms_total,usa_oro_milk,usa_oro_balance,usa_silo_maiz,usa_silo_avena,usa_ensilado,usa_heno,usa_triticale,usa_melaza,usa_soya
0,2025-01-01,dieta Mayo,154,0.5,5,23.7259,50.21,2.491,1,0,0,0,0,0,1,1,0
1,2025-01-02,dieta Mayo,154,0.5,5,23.7259,50.21,2.491,1,0,0,0,0,0,1,1,0
2,2025-01-03,dieta Mayo,154,0.5,5,23.7259,50.21,2.491,1,0,0,0,0,0,1,1,0
3,2025-01-04,dieta Mayo,154,0.5,5,23.7259,50.21,2.491,1,0,0,0,0,0,1,1,0
4,2025-01-05,dieta Mayo,154,0.5,5,23.7259,50.21,2.491,1,0,0,0,0,0,1,1,0


### 7.7 Dieta detallada por ingrediente expandida desde `diet_periods`

In [13]:

diet_periods_expanded = []

for _, row in diet_periods_raw.iterrows():
    start = row.get("period_start")
    end = row.get("period_end")
    if pd.isna(start) or pd.isna(end):
        continue

    dates = pd.date_range(start, end, freq="D")
    tmp = pd.DataFrame({
        "date": dates,
        "ingrediente": row.get("ingrediente"),
        "kg_humeda": row.get("kg_humeda"),
        "kg_seca": row.get("kg_seca"),
        "pct_ms": row.get("pct_ms"),
        "kg_am_pm": row.get("kg_am_pm"),
        "diet_period": row.get("diet_period"),
    })
    diet_periods_expanded.append(tmp)

diet_periods_daily_long = (
    pd.concat(diet_periods_expanded, ignore_index=True)
    if diet_periods_expanded
    else pd.DataFrame(columns=["date", "ingrediente"])
)

if not diet_periods_daily_long.empty:
    diet_periods_daily_long["ingrediente_clean"] = diet_periods_daily_long["ingrediente"].map(sanitize_name)

    diet_kg_seca = (
        diet_periods_daily_long
        .pivot_table(index="date", columns="ingrediente_clean", values="kg_seca", aggfunc="first")
        .add_prefix("diet_kg_seca_")
        .reset_index()
    )

    diet_kg_humeda = (
        diet_periods_daily_long
        .pivot_table(index="date", columns="ingrediente_clean", values="kg_humeda", aggfunc="first")
        .add_prefix("diet_kg_humeda_")
        .reset_index()
    )

    diet_pct_ms = (
        diet_periods_daily_long
        .pivot_table(index="date", columns="ingrediente_clean", values="pct_ms", aggfunc="first")
        .add_prefix("diet_pct_ms_")
        .reset_index()
    )

    diet_kg_am_pm = (
        diet_periods_daily_long
        .pivot_table(index="date", columns="ingrediente_clean", values="kg_am_pm", aggfunc="first")
        .add_prefix("diet_kg_am_pm_")
        .reset_index()
    )

    diet_detail_daily = diet_kg_seca.merge(diet_kg_humeda, on="date", how="outer")
    diet_detail_daily = diet_detail_daily.merge(diet_pct_ms, on="date", how="outer")
    diet_detail_daily = diet_detail_daily.merge(diet_kg_am_pm, on="date", how="outer")
else:
    diet_detail_daily = pd.DataFrame(columns=["date"])

print("diet_detail_daily ->", diet_detail_daily.shape)
display(diet_detail_daily.head())


diet_detail_daily -> (273, 41)


ingrediente_clean,date,diet_kg_seca_agua,diet_kg_seca_maiz_molido,diet_kg_seca_melaza50pct_agua,diet_kg_seca_oro_balance,diet_kg_seca_oro_milk,diet_kg_seca_pasta_se_soya,diet_kg_seca_pata_de_cebada,diet_kg_seca_silo_de_avena,diet_kg_seca_silo_de_maiz,...,diet_kg_am_pm_agua,diet_kg_am_pm_maiz_molido,diet_kg_am_pm_melaza50pct_agua,diet_kg_am_pm_oro_balance,diet_kg_am_pm_oro_milk,diet_kg_am_pm_pasta_se_soya,diet_kg_am_pm_pata_de_cebada,diet_kg_am_pm_silo_de_avena,diet_kg_am_pm_silo_de_maiz,diet_kg_am_pm_triticale
0,2025-01-01,0.0055,NaN,1.332,NaN,14.184,NaN,0.4048,NaN,NaN,...,423.5,NaN,256.41,NaN,1213.52,NaN,35.42,NaN,NaN,1937.32
1,2025-01-02,0.0055,NaN,1.332,NaN,14.184,NaN,0.4048,NaN,NaN,...,423.5,NaN,256.41,NaN,1213.52,NaN,35.42,NaN,NaN,1937.32
2,2025-01-03,0.0055,NaN,1.332,NaN,14.184,NaN,0.4048,NaN,NaN,...,423.5,NaN,256.41,NaN,1213.52,NaN,35.42,NaN,NaN,1937.32
3,2025-01-04,0.0055,NaN,1.332,NaN,14.184,NaN,0.4048,NaN,NaN,...,423.5,NaN,256.41,NaN,1213.52,NaN,35.42,NaN,NaN,1937.32
4,2025-01-05,0.0055,NaN,1.332,NaN,14.184,NaN,0.4048,NaN,NaN,...,423.5,NaN,256.41,NaN,1213.52,NaN,35.42,NaN,NaN,1937.32


## 8. Construcción de la llave base `cow_id + date`

In [14]:

# Base = unión de las fuentes que describen observaciones a nivel vaca-fecha.
# Así no se pierden fechas donde solo existe rumia o solo existen eventos.
base_keys = (
    pd.concat(
        [
            milking_daily[["cow_id", "date"]],
            rumination_daily[["cow_id", "date"]],
            events_daily[["cow_id", "date"]],
        ],
        ignore_index=True,
    )
    .dropna(subset=["cow_id", "date"])
    .drop_duplicates()
    .sort_values(["date", "cow_id"])
    .reset_index(drop=True)
)

print("base_keys ->", base_keys.shape)
print("Vacas únicas :", base_keys["cow_id"].nunique())
print("Fechas únicas:", base_keys["date"].nunique())
print("Rango fechas :", base_keys["date"].min(), "->", base_keys["date"].max())
display(base_keys.head())


base_keys -> (18345, 2)
Vacas únicas : 78
Fechas únicas: 999
Rango fechas : 2021-04-03 00:00:00 -> 2025-12-31 00:00:00


,cow_id,date
0,1638,2021-04-03
1,1644,2021-04-26
2,2074,2021-06-18
3,2076,2021-06-29
4,2082,2021-07-21


## 9. Merge principal

In [15]:

merged = base_keys.copy()

merged = merged.merge(milking_daily, on=["cow_id", "date"], how="left")
merged = merged.merge(rumination_daily, on=["cow_id", "date"], how="left")
merged = merged.merge(events_daily, on=["cow_id", "date"], how="left")
merged = merged.merge(weather_daily, on="date", how="left")
merged = merged.merge(feeding_daily, on="date", how="left")
merged = merged.merge(diet_summary_daily, on="date", how="left", suffixes=("", "_diet_summary"))
merged = merged.merge(diet_detail_daily, on="date", how="left")

# Derivados simples
if "event_count" in merged.columns:
    merged["event_count"] = merged["event_count"].fillna(0).astype("int64")
    merged["has_event"] = merged["event_count"].gt(0).astype("int64")

merged = merged.sort_values(["cow_id", "date"]).reset_index(drop=True)

print("merged ->", merged.shape)
display(merged.head())


merged -> (18345, 175)


,cow_id,date,ordenos_dia,produccion_kg,duracion_total_min,duracion_prom_min,intervalo_ordeno_prom_min,di_mean,dd_mean,td_mean,...,diet_kg_am_pm_maiz_molido,diet_kg_am_pm_melaza50pct_agua,diet_kg_am_pm_oro_balance,diet_kg_am_pm_oro_milk,diet_kg_am_pm_pasta_se_soya,diet_kg_am_pm_pata_de_cebada,diet_kg_am_pm_silo_de_avena,diet_kg_am_pm_silo_de_maiz,diet_kg_am_pm_triticale,has_event
0,1204,2022-06-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,1204,2022-06-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,1204,2022-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,1204,2022-08-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,1204,2022-09-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


## 10. Validaciones importantes

In [16]:

dup_count = merged.duplicated(subset=["cow_id", "date"]).sum()
print("Duplicados en cow_id + date:", dup_count)

print("\nCobertura:")
if "produccion_kg" in merged.columns:
    print("Filas con produccion_kg:", merged["produccion_kg"].notna().sum())
if "rumia_min" in merged.columns:
    print("Filas con rumia_min:", merged["rumia_min"].notna().sum())
if "event_count" in merged.columns:
    print("Filas con eventos:", merged["event_count"].gt(0).sum())

for c in ["temperature_2m_mean", "kg_totales_total", "diet_period"]:
    if c in merged.columns:
        print(f"Filas con {c}:", merged[c].notna().sum())

print("\nRango de fechas final:", merged["date"].min(), "->", merged["date"].max())
print("Número de vacas:", merged["cow_id"].nunique())
print("Fechas únicas:", merged["date"].nunique())

assert dup_count == 0, "Existen duplicados en la llave cow_id + date"


Duplicados en cow_id + date: 0

Cobertura:
Filas con produccion_kg: 9814
Filas con rumia_min: 5750
Filas con eventos: 4025
Filas con temperature_2m_mean: 18345
Filas con kg_totales_total: 15648
Filas con diet_period: 15767

Rango de fechas final: 2021-04-03 00:00:00 -> 2025-12-31 00:00:00
Número de vacas: 78
Fechas únicas: 999


## 11. Reporte de faltantes

In [17]:

missing_report = (
    pd.DataFrame({
        "column": merged.columns,
        "dtype": [str(merged[c].dtype) for c in merged.columns],
        "nan_count": merged.isna().sum().values,
        "nan_pct": merged.isna().mean().mul(100).round(2).values,
    })
    .sort_values(["nan_pct", "nan_count"], ascending=[False, False])
    .reset_index(drop=True)
)

display(missing_report.head(50))


,column,dtype,nan_count,nan_pct
0,weight_kg,float64,18343,99.99
1,del_at_dryoff,float64,18340,99.97
2,body_condition_score,float64,18331,99.92
3,gestant_insem_number,float64,18312,99.82
4,liters_at_dryoff,float64,18304,99.78
5,dcc_at_dryoff,float64,18297,99.74
6,diet_kg_seca_maiz_molido,float64,14627,79.73
7,diet_kg_seca_pasta_se_soya,float64,14627,79.73
8,diet_kg_seca_silo_de_avena,float64,14627,79.73
9,diet_kg_seca_silo_de_maiz,float64,14627,79.73


## 12. Guardado

In [18]:

pq.write_table(pa.Table.from_pandas(merged, preserve_index=False), OUT_DATASET)
missing_report.to_csv(OUT_MISSING, index=False)

print("Archivo principal guardado en:")
print(OUT_DATASET.resolve())

print("\nReporte de faltantes guardado en:")
print(OUT_MISSING.resolve())


Archivo principal guardado en:
/Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/processed/training_dataset.parquet

Reporte de faltantes guardado en:
/Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/processed/training_dataset_missing_report.csv



## 13. Nota final

Este notebook genera un dataset listo para análisis con una fila por:

`(cow_id, date)`

### Importante
- `events.parquet` sirve para auditoría detallada.
- `events_daily.parquet` es la versión correcta para hacer el merge analítico.
- `feeding_daily_corral.parquet` se agregó por **fecha**, no por vaca, porque todavía no hay un mapeo confiable vaca-corral por día.
